# EduYou — RAG Query Notebook (Azure OpenAI)

**File:** `rag_query_eduyou.ipynb`  
**Generated:** 2026-03-30

This notebook mirrors the professor’s `07_RAG_query.ipynb` template and is adapted for the EduYou capstone.

It:
1. Loads precomputed document embeddings
2. Embeds a user query using the **same embedding model**
3. Computes cosine similarity
4. Retrieves Top‑K documents
5. (Optional) passes retrieved text to an LLM

---

## Required inputs
- Document table: `cleaned/eduyou_cip_docs_for_embedding.csv`
- Embeddings table: `embeddings/eduyou_embeddings_<MODEL>.csv`

> ⚠️ Both must be generated using the **same embedding model**.


In [ ]:
import os
import numpy as np
import pandas as pd
from openai import AzureOpenAI
from sklearn.metrics.pairwise import cosine_similarity


## 1) Configuration

Ensure the same Azure OpenAI embedding deployment is used here as in `get_embeddings_eduyou.ipynb`.


In [ ]:
# Azure OpenAI configuration
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_KEY  = os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_API_VERSION = '2024-02-01'

# Must match embeddings notebook
AZURE_OPENAI_DEPLOYMENT_ID = os.getenv('AZURE_OPENAI_DEPLOYMENT_ID', 'text-embedding-3-small')

# File paths
DOCS_CSV = 'cleaned/eduyou_cip_docs_for_embedding.csv'
EMB_CSV  = f'embeddings/eduyou_embeddings_{AZURE_OPENAI_DEPLOYMENT_ID}.csv'

TOP_K = 5


## 2) Load documents and embeddings


In [ ]:
docs = pd.read_csv(DOCS_CSV, low_memory=False)
embs = pd.read_csv(EMB_CSV, low_memory=False)

# Ensure alignment via doc_id
assert set(docs['doc_id']) == set(embs['doc_id']), 'Mismatch between docs and embeddings'

# Extract embedding matrix
embed_cols = [c for c in embs.columns if c.startswith('dim_')]
X = embs[embed_cols].values

print('Documents:', len(docs))
print('Embedding dimension:', X.shape[1])


## 3) Embed a user query


In [ ]:
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

query = "What careers and salaries are associated with Communication and Media Studies?"

q_resp = client.embeddings.create(
    model=AZURE_OPENAI_DEPLOYMENT_ID,
    input=query
)

q_vec = np.array(q_resp.data[0].embedding).reshape(1, -1)
print('Query embedded.')


## 4) Retrieve Top‑K documents by cosine similarity


In [ ]:
sims = cosine_similarity(q_vec, X)[0]

idx = np.argsort(sims)[::-1][:TOP_K]
results = docs.iloc[idx].copy()
results['similarity'] = sims[idx]

results[['doc_id','cip_title','degree_level','similarity']]


## 5) Inspect retrieved document text


In [ ]:
for _, row in results.iterrows():
    print('---')
    print(f"{row['doc_id']} (score={row['similarity']:.3f})")
    print(row['text'][:1200])


## 6) (Optional) Pass retrieved context to an LLM

This mirrors the professor’s RAG pattern: retrieved documents are concatenated and provided as context to a generation model.


In [ ]:
context = '

'.join(results['text'].astype(str).tolist())

prompt = (
    "You are an educational assistant.

"
    "Using the information below, answer the user question clearly and accurately.

"
    f"Context:
{context}

"
    f"Question:
{query}
"
)

# Example (only if a chat model is deployed):
# completion = client.chat.completions.create(
#     model='gpt-4o-mini',
#     messages=[{'role': 'user', 'content': prompt}]
# )
# print(completion.choices[0].message.content)


## Summary

- ✅ One row = one document
- ✅ Same embedding model for docs and query
- ✅ Cosine similarity retrieval
- ✅ Compatible with professor’s RAG template
